# SmartFarm ML — Stage 03: Honest evaluation
**Goal:** prove a model is good — or admit it isn't. Accuracy alone lies; here we use
the confusion matrix, precision / recall / F1, and cross-validation, then run the
proper showdown against the crop-aware rules baseline. Data: `irrigation_honest.csv`.

## 1. Setup — same honest data, same split

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv("irrigation_honest.csv")
feature_cols = ["soil_moisture", "air_humidity", "temperature"]
X, y = df[feature_cols], df["irrigate"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

## 2. The accuracy trap
Our data is 58% "no water". A lazy model that ALWAYS says "no water" scores 58% —
without ever making a single useful decision. So compare any accuracy against this floor.

In [ ]:
lazy_acc  = (y_test == 0).mean()          # always-predict-0 accuracy
model_acc = model.score(X_test, y_test)
print(f"lazy 'always no-water' accuracy: {lazy_acc:.3f}")
print(f"logistic regression accuracy:   {model_acc:.3f}")

## 3. Confusion matrix — WHERE the model is wrong
`.predict()` gives 0/1 for each test row. The confusion matrix buckets those into 4 cells.
Layout: rows = actual, columns = predicted. Label 1 = "irrigate".

```
                 pred: no-water   pred: water
actual no-water      TN               FP        <- FP = watered when not needed (wasted water)
actual water         FN               TP        <- FN = did NOT water when needed (crop goes thirsty!)
```

In [ ]:
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)   # [[TN, FP],[FN, TP]]
print(cm)

tn, fp, fn, tp = cm.ravel()             # flatten the 2x2 into 4 numbers, in reading order
print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"\nFP (wasted water):      {fp}  -> minor: some water wasted")
print(f"FN (missed thirsty crop): {fn}  -> serious: crop left dry")

## 4. Precision, Recall, F1 — the errors as meaningful numbers
- **precision** = of the times the model SAID water, how many truly needed it = `TP/(TP+FP)`.
  Low precision → wastes water.
- **recall** = of all the times water WAS truly needed, how many did the model catch = `TP/(TP+FN)`.
  Low recall → misses thirsty crops. **In irrigation this is the dangerous one.**
- **F1** = single blended score of the two (harmonic mean).

In [ ]:
print(f"precision: {precision_score(y_test, y_pred):.3f}")
print(f"recall:    {recall_score(y_test, y_pred):.3f}")
print(f"f1:        {f1_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["no_water","water"]))

## 5. One split can fool you — cross-validation
A single train/test split is one roll of the dice. `cross_val_score(cv=5)` splits the data
5 different ways, trains 5 times, and reports 5 scores. Look at the **mean** (true level)
and the **std** (how much it wobbles). If std is large, don't trust any single number.

In [ ]:
scores = cross_val_score(model, X, y, cv=5)     # 5 fits on 5 different folds
print("per-fold accuracy:", scores.round(3))
print(f"mean: {scores.mean():.3f}   std: {scores.std():.3f}")
print(f"(single-split accuracy earlier was {model_acc:.3f} - a bit lucky vs the mean)")

## 6. The real showdown — model vs crop-aware rules, on the metric that matters
Accuracy hid this. Compare on **recall** (catching thirsty crops), the failure that kills crops.

In [ ]:
ideal = {"tomato": 58, "chili": 45, "okra": 50}
base_pred = (df.loc[X_test.index, "soil_moisture"]
             < df.loc[X_test.index, "crop_type"].map(ideal)).astype(int)

print(f"{'':12s}{'precision':>10s}{'recall':>10s}{'f1':>8s}")
print(f"{'model':12s}{precision_score(y_test,y_pred):>10.3f}{recall_score(y_test,y_pred):>10.3f}{f1_score(y_test,y_pred):>8.3f}")
print(f"{'rules':12s}{precision_score(y_test,base_pred):>10.3f}{recall_score(y_test,base_pred):>10.3f}{f1_score(y_test,base_pred):>8.3f}")

## The honest verdict
- On **accuracy** the model (0.78) edges the rules — looked like a win in Stage 02.
- But on **recall**, the rules baseline is HIGHER: it catches more thirsty crops. And on **F1**
  the rules also come out ahead. For irrigation, where missing a dry crop is the costly error,
  **the simple rules are the better tool right now.**
- This is the whole roadmap philosophy proven with numbers: *the model has not earned its place.*
  The thing it's missing is `crop_type` — Stage 05.

## Your turn
1. In farm terms, which is worse for a tomato plant: one FP or one FN? Does that make you care
   more about precision or recall? Write one sentence.
2. Re-run cross-validation with `cv=10`. Does the mean move? Does the std shrink or grow?
3. (Think) If you wanted the model to catch MORE thirsty crops (raise recall) even at the cost
   of wasting some water (lower precision), would you want it to predict "water" more often or
   less often? (We'll actually do this with thresholds in a later stage.)